[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LSDtopotools/lsdtt3_notebooks/blob/main/channel_extraction/apennines_channel_extraction.ipynb)

# Extracting a channel network in the northern Apennines

In this notebook we extract a river network from a digital elevation model (DEM) of the northern Apennines, in the valleys upstream (south and southwest) of Bologna, Italy. The area covers parts of the Reno, Setta and Savena catchments.

The workflow is:
1. Install `lsdtt3` (the command-line analysis tools) and `lsdviztools3` (the plotting package).
2. Download a 30 m DEM with `lsdtt-fetch-raster`.
3. Make a hillshade with `lsdtt-raster-preprocessing`.
4. Extract the channel network with `lsdtt-channel-extraction`.
5. Plot the channels, coloured by stream order, over the hillshade.

Each `lsdtt3` program is controlled by a small **parameter file**: a text file of `key: value` lines. We write these files from Python and then run the programs.

## Setup

### First set up condacolab

This is a bit of an annoying step that takes around 2 minutes.

**Note:** `condacolab.install()` restarts the Colab runtime. You will see a message saying the session crashed; this is expected. Just carry on running the cells below.

In [ ]:
!pip install -q condacolab

In [ ]:
import condacolab
condacolab.install()

Now we install `pygmt`. GMT stands for Generic Mapping Tools, and `lsdviztools3` uses it to make maps. This is the slowest step (around a minute).

In [ ]:
!mamba install pygmt

### Get lsdviztools3

In [ ]:
!wget https://www.geos.ed.ac.uk/~smudd/lsdtt_packages/lsdviztools3-0.1.0-py3-none-any.whl

In [ ]:
!pip install lsdviztools3-0.1.0-py3-none-any.whl

### Get the lsdtt3 command-line tools

In [ ]:
!wget https://www.geos.ed.ac.uk/~smudd/lsdtt_packages/lsdtt3-backend-linux-x86_64-core-v0.5.2.tar.gz

In [ ]:
!tar -xzf lsdtt3-backend-linux-x86_64-core-v0.5.2.tar.gz

Now we tell the system where to find the `lsdtt3` programs and the libraries they need. If your runtime restarts later, run this cell again.

In [ ]:
import os

root = "/content/lsdtt3-backend-linux-x86_64-core-v0.5.2"
os.environ["PATH"] = f"{root}/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = f"{root}/lib:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["GDAL_DATA"] = f"{root}/share/gdal"
os.environ["PROJ_DATA"] = f"{root}/share/proj"
os.environ["PROJ_LIB"]  = os.environ["PROJ_DATA"]

Check that it works: this prints the version of `lsdtt3`.

In [ ]:
!lsdtt-channel-extraction -v

## Step 1: Download a DEM

`lsdtt-fetch-raster` downloads a DEM for a bounding box given in longitude and latitude (WGS84 degrees). We use:

* `dem_source: cop30_aws`: the Copernicus GLO-30 DEM (about 30 m resolution), served from AWS. No account or API key is needed.
* `target_epsg: 32632`: reproject to **UTM zone 32N**, a projected coordinate system in metres. Flow routing needs distances in metres, not degrees. (If you leave this out, `lsdtt-fetch-raster` picks the UTM zone from the centre of the box, which here is also zone 32N.)
* `grid_spacing: 30`: a 30 m output grid.

Our box (longitude 11.0 to 11.4, latitude 44.15 to 44.45) is roughly 32 km by 34 km, about 1100 by 1100 pixels, so it downloads in a few seconds.

The output is called `<write_fname>_DEM.tif`, so here `apennines_DEM.tif`.

In [ ]:
# Write the parameter file for lsdtt-fetch-raster
with open("apennines_fetch.param", "w") as f:
    f.write("write_fname: apennines\n")
    f.write("west: 11.0\n")
    f.write("east: 11.4\n")
    f.write("south: 44.15\n")
    f.write("north: 44.45\n")
    f.write("dem_source: cop30_aws\n")
    f.write("target_epsg: 32632\n")
    f.write("grid_spacing: 30\n")

# Run it. The first argument is the directory holding the parameter file.
!lsdtt-fetch-raster ./ apennines_fetch.param

## Step 2: Make a hillshade

A hillshade shows the topography as if lit by the sun, which makes a good background map. `lsdtt-raster-preprocessing` fills pits in the DEM (`write_fill_raster`) and then computes the hillshade from the filled DEM (`write_hillshade_raster`). The sun is in the northwest (`hillshade_azimuth: 315`), 45 degrees above the horizon (`hillshade_altitude: 45`).

Outputs: `apennines_fill.tif` and `apennines_hillshade.tif`.

In [ ]:
with open("apennines_hillshade.param", "w") as f:
    f.write("read_fname: apennines_DEM.tif\n")
    f.write("write_fname: apennines\n")
    f.write("write_fill_raster: true\n")
    f.write("write_hillshade_raster: true\n")
    f.write("hillshade_azimuth: 315\n")
    f.write("hillshade_altitude: 45\n")

!lsdtt-raster-preprocessing ./ apennines_hillshade.param

## Step 3: Extract the channel network

`lsdtt-channel-extraction` fills pits in the DEM, routes flow with the D8 method (each pixel drains to its steepest downhill neighbour), finds channel heads (sources), and builds the network downstream from them.

We use the **threshold** method (`source_extraction_algorithm_choice: threshold`): a channel starts wherever the number of upslope pixels draining to a point reaches `threshold_contributing_pixels`. We set this to 500. With 30 m pixels each pixel is 900 m², so the channels start where the drainage area reaches 500 x 900 m² = 0.45 km².

Why this method and value?
* A 30 m DEM is too coarse to see real channel heads, which are often only a few metres wide. Methods that look for channel heads in the shape of the topography (such as the `wiener` option) are designed for high-resolution lidar data.
* The threshold is a choice, not a measurement: 500 pixels gives a network that fills the valleys without drawing channels on every hillslope. Try 200 or 2000 and see what changes.

We ask for three outputs:
* `apennines_channel_network_lines.fgb`: one line per channel link (between junctions), with a `stream_order` attribute (Strahler order). We plot this.
* `apennines_channel_network_points.csv`: one row per channel pixel with `latitude`, `longitude` and `stream_order`, handy for other software.
* `apennines_channel_network_junctions.fgb`: the junctions where channels meet.

In [ ]:
with open("apennines_channels.param", "w") as f:
    f.write("read_fname: apennines_DEM.tif\n")
    f.write("write_fname: apennines\n")
    f.write("source_extraction_algorithm_choice: threshold\n")
    f.write("threshold_contributing_pixels: 500\n")
    f.write("write_channel_network_lines: true\n")
    f.write("write_channel_network_points: true\n")
    f.write("channel_network_points_write_vector_format: csv\n")
    f.write("write_channel_network_junctions: true\n")

!lsdtt-channel-extraction ./ apennines_channels.param

Let's look at the files we have made, and at the first few lines of the channel points file.

In [ ]:
!ls -lh apennines_*

import pandas as pd
points = pd.read_csv("apennines_channel_network_points.csv")
points.head()

## Step 4: Plot the channels over the hillshade

We use `lsdviztools3`:
* `load_vector` reads the channel lines into a GeoPandas table.
* `render_hillshade` draws the hillshade. `shade=False` because the raster is already a hillshade, and we use a grey colour map.
* `render_channels` draws channel lines on top of an existing figure (`fig=fig`).

To show **stream order**, we draw each order separately: higher-order (bigger) rivers get thicker, darker lines. In Strahler ordering, two order-1 streams join to make an order-2 stream, two order-2 streams make an order-3 stream, and so on.

In [ ]:
from dataclasses import replace

from lsdviztools3.io.vector import load_vector
from lsdviztools3.render.style import MapStyle
from lsdviztools3.render.hillshade import render_hillshade
from lsdviztools3.render.channels import render_channels
from lsdviztools3.render.base import save_figure

lines = load_vector("apennines_channel_network_lines.fgb")
print("Coordinate system:", lines.crs)
print("Number of channel links in each stream order:")
print(lines["stream_order"].value_counts().sort_index())

In [ ]:
style = MapStyle(figure_size="15c", transparent=False)

# Grey hillshade base map
fig = render_hillshade("apennines_hillshade.tif", shade=False,
                       style=replace(style, cmap="gray"))

# Channels: one layer per stream order, thicker and darker for bigger rivers
colours = {1: "lightskyblue", 2: "deepskyblue", 3: "dodgerblue",
           4: "blue", 5: "navy", 6: "midnightblue"}
for order in sorted(lines["stream_order"].unique()):
    subset = lines[lines["stream_order"] == order]
    order_style = replace(style,
                          line_width=f"{0.4 * order:.1f}p",
                          line_color=colours.get(int(order), "black"))
    fig = render_channels(subset, style=order_style, fig=fig)

save_figure(fig, "apennines_channels.png", style=style)

In [ ]:
from IPython.display import Image, display
display(Image("apennines_channels.png", width=700))

## Things to try

* Change `threshold_contributing_pixels` (for example to 200 or 2000) and rerun Step 3 and Step 4. How does the network change?
* Move the bounding box in Step 1 to another mountain range. Keep it small (well under a degree on each side) so it runs quickly.
* Colour the channels by a single attribute instead: `render_channels(lines, color_by="drainage_area", colorbar=True, fig=fig)` colours each link by the log of its drainage area (slower, because every link is drawn separately).